# Docs Q&A Bot -- RAG Core Demo (Retrieval + Generation, No Discord)

This notebook is a companion to the course project
[Build a RAG-Backed Docs Q&A Discord Bot](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/docs-qa-bot).
It demos the **retrieval-augmented generation (RAG) core** of that project -- embedding a small docs folder,
retrieving the most relevant chunks for a question, and asking an LLM to answer using only that retrieved context --
the exact same `build_index.py` / `retrieve.py` / `answer()` logic the real project uses.

**What this notebook deliberately does NOT do: run the actual Discord bot.** The real deliverable of that project is a
live `discord.py` bot -- a long-running process that holds an open connection to Discord's Gateway and waits
indefinitely for `@mention` events. Colab, Kaggle, and Binder are all built around running a cell, getting output,
and moving on to the next cell; none of them can host a background process that stays up once you close the tab or
the runtime recycles. That part of the project genuinely needs a real local process (or GitHub Codespaces) -- see
[the full lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/docs-qa-bot#step-3-wire-the-bots-message-handler)
for `bot.py` and how to run it for real.

What you'll do here instead: embed the same three sample docs the real project ships with, build an in-memory
index, retrieve relevant chunks for a few example questions, and generate grounded answers -- everything the bot
does *except* the Discord wiring around it.

## 1. Install dependencies

No `discord.py` needed here -- just the retrieval and generation libraries.

In [ ]:
!pip install -q sentence-transformers numpy openai

## 2. The sample docs

These are the exact same three files the real project ships in `examples/docs-qa-bot/docs/` -- embedded directly
as strings here so the notebook needs no file upload or repo clone to run standalone.

In [ ]:
SAMPLE_DOCS = {
    "course-structure.md": "## Course structure\n\nThe course has two required sections, taught entirely in the browser via\nJupyterLite: Python 101, covering variables, loops, functions, and basic\ndata structures, and Data Analysis, covering pandas, DataFrames, and\ngroupby-style aggregation.\n\nPython 101 has two tracks: a Normal track for beginners and a Hard track\nthat goes deeper, including building a tiny language model from scratch out\nof word counts and bigram probabilities.\n\nAfter both sections, there is an optional, ungraded \"Real-World Projects\"\nsection. Unlike the rest of the course, these projects install Python for\nreal on your own machine using `uv`, instead of running in a browser\nsandbox.",
    "discord-bot-basics.md": "## Discord bot basics\n\nA Discord bot is a normal application that connects to Discord's Gateway\n(a persistent WebSocket connection) using a bot token generated in the\nDiscord Developer Portal. Once connected, it receives events -- a message\nbeing sent, a reaction being added, a member joining -- and can react to\nthem by calling Discord's REST API, most commonly to send a message back.\n\nThe `discord.py` library wraps this connection in a Python `Client` (or the\nhigher-level `commands.Bot`) with decorator-based event handlers. The two\nmost common ones are `on_message`, which fires for every message the bot\ncan see, and `on_ready`, which fires once when the bot finishes connecting.\n\nBots need explicit permission to read message content. This is a\n\"privileged intent\" that must be turned on in the Developer Portal under\nBot settings, in addition to being requested in code via `intents.message_content\n= True` -- forgetting the portal toggle is one of the most common reasons a\nbot silently receives empty message content.",
    "rag-recap.md": "## RAG recap\n\nRetrieval-augmented generation (RAG) answers a question in two stages\ninstead of one. First, retrieval: embed the question into a vector, and\ncompare it against the vectors of every chunk of your documentation using\ncosine similarity, keeping only the closest few. Second, generation: hand\nthose retrieved chunks to a language model as context and ask it to answer\nusing only what it was given, rather than relying on what it already\nhappens to know.\n\nThis project reuses exactly that two-stage pipeline from the course's\n\"Build a RAG App Over Your Own Notes\" project -- the same local embedding\nmodel, the same NumPy cosine-similarity search, the same \"answer using only\nthis context\" prompt -- and swaps the CLI script that calls it for a\nDiscord bot's message handler instead."
}

for name, text in SAMPLE_DOCS.items():
    print(f"{name}: {len(text)} characters")

## 3. Chunk the docs

Identical chunking approach to `prepare_docs.py` in the real project: split on blank lines into paragraphs,
then greedily merge consecutive short paragraphs up to a target size so a chunk isn't just one short line with
no context. Only the input source changes -- an in-memory dict here instead of `.md` files on disk.

In [ ]:
TARGET_CHUNK_SIZE = 500  # characters


def split_into_paragraphs(text):
    paragraphs = [p.strip() for p in text.split("\n\n")]
    return [p for p in paragraphs if p]


def merge_short_paragraphs(paragraphs, target_size):
    chunks = []
    current = ""
    for paragraph in paragraphs:
        if current and len(current) + len(paragraph) > target_size:
            chunks.append(current)
            current = paragraph
        else:
            current = f"{current}\n\n{paragraph}" if current else paragraph
    if current:
        chunks.append(current)
    return chunks


def load_chunks(docs):
    chunks = []
    for source, text in docs.items():
        paragraphs = split_into_paragraphs(text)
        for chunk_text in merge_short_paragraphs(paragraphs, TARGET_CHUNK_SIZE):
            chunks.append({"text": chunk_text, "source": source})
    return chunks


chunks = load_chunks(SAMPLE_DOCS)
print(f"Loaded {len(chunks)} chunks from {len(SAMPLE_DOCS)} sample docs")
for c in chunks:
    preview = c["text"][:70].replace("\n", " ")
    print(f"  [{c['source']}] {preview}...")

## 4. Embed the chunks (build the index)

Identical logic to `build_index.py` -- embed every chunk locally with `sentence-transformers` and normalize
the vectors so cosine similarity collapses to a plain dot product later. No API key needed for this step;
embeddings run entirely on this notebook's CPU/GPU.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, normalize_embeddings=True)

print(f"Embedded {embeddings.shape[0]} chunks into {embeddings.shape[1]}-dim vectors")

## 5. Retrieve relevant chunks

Identical retrieval logic to `retrieve.py`: embed the question with the same model, then rank every chunk by
cosine similarity -- a plain dot product since every vector is already unit-length.

In [ ]:
def retrieve(question, top_k=3):
    question_vector = model.encode([question], normalize_embeddings=True)[0]
    similarities = embeddings @ question_vector
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [
        {**chunks[i], "score": float(similarities[i])}
        for i in top_indices
    ]


# Quick sanity check before wiring up generation
for r in retrieve("How do I enable the message content intent?"):
    print(f"{r['score']:.3f}  [{r['source']}]  {r['text'][:80]}...")

## 6. Get a free-tier LLM API key

Generation needs one free-tier LLM key -- same options as the lesson's table. **GitHub Models** is the
suggested default (a GitHub personal access token with the `models: read` scope, from
[github.com/settings/tokens](https://github.com/settings/tokens), no separate signup). Gemini, Groq, Mistral,
Cerebras, and OpenRouter all work too -- see the lesson for details on each.

The cell below uses `getpass` so the key is never echoed to the notebook's output or saved in its cell history.

In [ ]:
from getpass import getpass

llm_api_key = getpass("Paste your GitHub Models token (or other provider key): ")

## 7. Generate an answer grounded in the retrieved chunks

Identical logic to `bot.py`'s `answer()` -- retrieve, build the "answer using only this context" prompt, call
the LLM -- just returning a string here instead of handing it to `message.reply(...)`. This block defaults to
GitHub Models' OpenAI-compatible endpoint; swap the `base_url` (and the key above) for your own provider's
client if you picked a different one.

In [ ]:
from openai import OpenAI

llm_client = OpenAI(
    api_key=llm_api_key,
    base_url="https://models.github.ai/inference",  # swap for your provider's base_url if different
)

PROMPT_TEMPLATE = """Answer the question using ONLY the context below. If the
context doesn't contain the answer, say so -- do not make something up.
Keep the answer concise.

Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(f"[{c['source']}] {c['text']}" for c in retrieved_chunks)
    return PROMPT_TEMPLATE.format(context=context, question=question)


def answer(question, top_k=3):
    retrieved_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved_chunks)
    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",  # confirm this still has a free tier before running
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

## 8. Try it end to end

A few real example questions against the sample docs -- one the docs clearly cover, one that needs a specific
detail, and one the docs don't cover at all (to check the model says so honestly instead of guessing).

In [ ]:
example_questions = [
    "What are the two required sections of the course?",
    "What privileged intent does a Discord bot need to read message content?",
    "What programming language does the course use for its capstone project?",  # not covered by these docs
]

for question in example_questions:
    print(f"Q: {question}")
    print(f"A: {answer(question)}")
    print()

## What this notebook did (and didn't) show

This covered the full RAG core: chunk sample docs, embed them locally, retrieve the most relevant chunks for a
question with cosine similarity, and generate a grounded answer with a free-tier LLM -- exactly the pipeline
`bot.py` calls from inside `on_message`.

What it did not show is the Discord layer itself: connecting to Discord's Gateway, staying online indefinitely,
and replying in a real server when someone `@mentions` the bot. That needs a genuine long-running process, which
is exactly what Colab/Kaggle/Binder sessions are not built for. To run the actual bot:

- Clone the repo and run `uv run python bot.py` locally (see the
[Setup](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/docs-qa-bot#setup) and
[Step 3](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/docs-qa-bot#step-3-wire-the-bots-message-handler)
sections of the lesson), or
- Open the repo in a free [GitHub Codespace](https://codespaces.new/abderrahim-lectures/python-data-analysis-course)
and run it there -- see the lesson's "Where to run this" section for details.